# Interactive Data Visualizer

This notebook provides an interactive visualization tool for exploring RGB images and point clouds from your HDF5 dataset. You can use sliders to navigate through different timesteps and view the corresponding data.

In [43]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import cv2
from IPython.display import display, clear_output
import ipywidgets as widgets


In [ ]:
# Replace with your HDF5 file path
import os
path = "/home/xinhai/automoma/output/test_collect/traj/summit_franka/scene_10_seed_10/7221/grasp_0001/camera_data/episode000001.hdf5"

# Check if file exists before opening
if not os.path.exists(path):
    raise FileNotFoundError(f"HDF5 file not found at: {path}")

# Open the HDF5 file with error handling
try:
    file = h5py.File(path, "r")
except Exception as e:
    print(f"Error opening HDF5 file: {e}")
    file = None

# Get number of timestep

In [45]:
def print_hdf5_structure(name, obj, indent=0):
    """Recursively print the structure of an HDF5 file"""
    prefix = "  " * indent
    if isinstance(obj, h5py.Group):
        print(f"{prefix}- {name} (group)")
        for key in obj.keys():
            print_hdf5_structure(key, obj[key], indent + 1)
    elif isinstance(obj, h5py.Dataset):
        print(f"{prefix}- {name} (dataset)")
        print(f"{prefix}  shape: {obj.shape}, dtype: {obj.dtype}")

# Print the structure of the HDF5 file
print("HDF5 File Structure:")
print("==================")
for key in file.keys():
    print_hdf5_structure(key, file[key])

HDF5 File Structure:
- env_info (group)
  - grasp_pose (dataset)
    shape: (), dtype: object
  - object_id (dataset)
    shape: (), dtype: object
  - robot_name (dataset)
    shape: (), dtype: object
  - scene_id (dataset)
    shape: (), dtype: object
- obs (group)
  - depth (group)
    - ego_topdown (dataset)
      shape: (32, 240, 320), dtype: float32
    - ego_wrist (dataset)
      shape: (32, 240, 320), dtype: float32
    - fix_local (dataset)
      shape: (32, 240, 320), dtype: float32
  - eef (dataset)
    shape: (32, 7), dtype: float32
  - joint (group)
    - arm (dataset)
      shape: (32, 7), dtype: float64
    - gripper (dataset)
      shape: (32,), dtype: float64
    - mobile_base (dataset)
      shape: (32, 3), dtype: float64
  - point_cloud (dataset)
    shape: (32, 4096, 6), dtype: float64
  - rgb (group)
    - ego_topdown (dataset)
      shape: (32, 240, 320, 3), dtype: uint8
    - ego_wrist (dataset)
      shape: (32, 240, 320, 3), dtype: uint8
    - fix_local (dataset

In [46]:
file["env_info"]["robot_name"][()].decode("utf-8")  # Get robot name

'summit_franka'

In [47]:
def decode_rgb_image(rgb_bytes):
    """Decode RGB image from byte string"""
    try:
        nparr = np.frombuffer(rgb_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        if img is not None:
            # Convert BGR to RGB
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            return img_rgb
    except Exception as e:
        print(f"Error decoding image: {e}")
    return None

def visualize_rgb_images(step):
    """Visualize RGB images for a given timestep"""
    if 'obs' not in file or 'rgb' not in file['obs']:
        print("No observation RGB data found")
        return
    
    rgb_group = file['obs']['rgb']
    cameras = ['ego_topdown', 'ego_wrist', 'fix_local']
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'RGB Images at Step {step}', fontsize=16)
    
    for i, camera in enumerate(cameras):
        if camera in rgb_group:
            try:
                img_rgb = rgb_group[camera][step]
                
                if img_rgb is not None:
                    axes[i].imshow(img_rgb)
                    axes[i].set_title(f'{camera}')
                    axes[i].axis('off')
                else:
                    axes[i].set_title(f'{camera} (decode failed)')
                    axes[i].axis('off')
            except Exception as e:
                axes[i].set_title(f'{camera} (error)')
                axes[i].axis('off')
        else:
            axes[i].set_title(f'{camera} (not found)')
            axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

def visualize_depth_images(step):
    """Visualize depth images for a given timestep"""
    if 'obs' not in file or 'depth' not in file['obs']:
        print("No observation depth data found")
        return
    
    depth_group = file['obs']['depth']
    cameras = ['ego_topdown', 'ego_wrist', 'fix_local']
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Depth Images at Step {step}', fontsize=16)
    
    for i, camera in enumerate(cameras):
        if camera in depth_group:
            try:
                depth_data = depth_group[camera][step]
                
                if depth_data is not None:
                    axes[i].imshow(depth_data, cmap='viridis')
                    axes[i].set_title(f'{camera}')
                    axes[i].axis('off')
                else:
                    axes[i].set_title(f'{camera} (decode failed)')
                    axes[i].axis('off')
            except Exception as e:
                axes[i].set_title(f'{camera} (error)')
                axes[i].axis('off')
        else:
            axes[i].set_title(f'{camera} (not found)')
            axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

def visualize_pointcloud(step, max_points=5000):
    """Visualize point cloud data"""
    if 'obs' not in file or 'point_cloud' not in file['obs']:
        print("No point cloud data found")
        return
    
    pc_data = file['obs']['point_cloud'][step]
    
    # Point cloud data is [N, 6] where first 3 are XYZ and last 3 are RGB
    xyz = pc_data[:, :3]
    rgb = pc_data[:, 3:]
    
    # Subsample for visualization if too many points
    if len(xyz) > max_points:
        indices = np.random.choice(len(xyz), max_points, replace=False)
        xyz = xyz[indices]
        rgb = rgb[indices]
    
    # Normalize RGB values to [0, 1] if needed
    if rgb.max() > 1.0:
        rgb = rgb / 255.0
    
    # Create 3D plot
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    fig.suptitle(f'Point Cloud at Step {step}', fontsize=16)
    
    if len(xyz) > 0:
        # Plot points with colors
        scatter = ax.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], 
                           c=rgb, s=0.8, alpha=0.6)
        
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'{len(xyz)} points')
    
    plt.show()

def visualize_eef_pose(step):
    """Visualize end effector pose data"""
    if 'obs' not in file or 'eef' not in file['obs']:
        print("No end effector pose data found")
        return
    
    eef_data = file['obs']['eef'][step]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'End Effector Pose at Step {step}', fontsize=16)
    
    # Position data (first 3 values)
    position = eef_data[:3]
    axes[0].bar(range(len(position)), position)
    axes[0].set_title('EEF Position (XYZ)')
    axes[0].set_ylabel('Position Values')
    axes[0].set_xlabel('Dimensions')
    axes[0].set_xticks(range(len(position)))
    axes[0].set_xticklabels(['X', 'Y', 'Z'])
    axes[0].grid(True)
    
    # Orientation data (last 4 values - quaternion)
    quaternion = eef_data[3:]
    axes[1].bar(range(len(quaternion)), quaternion)
    axes[1].set_title('EEF Orientation (Quaternion)')
    axes[1].set_ylabel('Quaternion Values')
    axes[1].set_xlabel('Dimensions')
    axes[1].set_xticks(range(len(quaternion)))
    axes[1].set_xticklabels(['X', 'Y', 'Z', 'W'])
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

def visualize_joint_states(step):
    """Visualize joint state data"""
    if 'obs' not in file or 'joint' not in file['obs']:
        print("No joint state data found")
        return
    
    joint_group = file['obs']['joint']
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 10))
    fig.suptitle(f'Joint State Data at Step {step}', fontsize=16)
    
    # Mobile base joint states
    if 'mobile_base' in joint_group:
        mobile_base = joint_group['mobile_base'][step]
        axes[0].bar(range(len(mobile_base)), mobile_base)
        axes[0].set_title('Mobile Base Joint States')
        axes[0].set_ylabel('Joint Values')
        axes[0].set_xlabel('Joint Index')
        axes[0].grid(True)
    else:
        axes[0].set_title('Mobile Base Joint States (Not Found)')
        axes[0].axis('off')

    # Arm joint states
    if 'arm' in joint_group:
        arm = joint_group['arm'][step]
        axes[1].bar(range(len(arm)), arm)
        axes[1].set_title('Arm Joint States')
        axes[1].set_ylabel('Joint Values')
        axes[1].set_xlabel('Joint Index')
        axes[1].grid(True)
    else:
        axes[1].set_title('Arm Joint States (Not Found)')
        axes[1].axis('off')

    
    # gripper states
    if 'gripper' in joint_group:
        gripper = joint_group['gripper'][step]
        axes[2].bar([0], [gripper])
        axes[2].set_title('Gripper State')
        axes[2].set_ylabel('Gripper Value')
        axes[2].set_xticks([0])
        axes[2].set_xticklabels(['Gripper'])
        axes[2].grid(True)
    else:
        axes[2].set_title('Gripper State (Not Found)')
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()

def visualize_action_eef(step):
    """Visualize action end effector data"""
    if 'action' not in file or 'eef' not in file['action']:
        print("No action EEF data found")
        return
    
    eef_data = file['action']['eef'][step]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Action EEF at Step {step}', fontsize=16)
    
    # Position data (first 3 values)
    position = eef_data[:3]
    axes[0].bar(range(len(position)), position)
    axes[0].set_title('Action EEF Position (XYZ)')
    axes[0].set_ylabel('Position Values')
    axes[0].set_xlabel('Dimensions')
    axes[0].set_xticks(range(len(position)))
    axes[0].set_xticklabels(['X', 'Y', 'Z'])
    axes[0].grid(True)
    
    # Orientation data (last 4 values - quaternion)
    quaternion = eef_data[3:]
    axes[1].bar(range(len(quaternion)), quaternion)
    axes[1].set_title('Action EEF Orientation (Quaternion)')
    axes[1].set_ylabel('Quaternion Values')
    axes[1].set_xlabel('Dimensions')
    axes[1].set_xticks(range(len(quaternion)))
    axes[1].set_xticklabels(['W', 'X', 'Y', 'Z'])
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()


In [48]:
# Get the number of steps from the dataset
num_steps = 32  # Default value
if 'obs' in file and 'rgb' in file['obs'] and 'ego_topdown' in file['obs']['rgb']:
    num_steps = file['obs']['rgb']['ego_topdown'].shape[0]
elif 'obs' in file and 'eef' in file['obs']:
    num_steps = file['obs']['eef'].shape[0]
print(f"Number of steps in dataset: {num_steps}")

def update_visualization(step):
    """Update visualization based on selected step"""
    clear_output(wait=True)
    
    # Show RGB images
    visualize_rgb_images(step)
    
    # Show depth images
    visualize_depth_images(step)
    
    # Show point cloud
    visualize_pointcloud(step)
    
    # Show end effector pose data
    visualize_eef_pose(step)
    
    # Show joint state data
    visualize_joint_states(step)

    
    print(f"Displaying data for step {step}")

# Create interactive slider
step_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_steps-1,
    step=1,
    description='Timestep:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='800px')
)

# Link slider to update function
interact_widget = widgets.interactive(update_visualization, step=step_slider)
display(interact_widget)

Number of steps in dataset: 32


interactive(children=(IntSlider(value=0, description='Timestep:', layout=Layout(width='800px'), max=31, style=…

In [49]:
# Alternative: Separate controls for different data types

def update_rgb_only(step):
    clear_output(wait=True)
    visualize_rgb_images(step)
    print(f"Displaying RGB images for step {step}")

def update_depth_only(step):
    clear_output(wait=True)
    visualize_depth_images(step)
    print(f"Displaying depth images for step {step}")

def update_pointcloud_only(step):
    clear_output(wait=True)
    visualize_pointcloud(step)
    print(f"Displaying point cloud for step {step}")
    
def update_eef_pose_only(step):
    clear_output(wait=True)
    visualize_eef_pose(step)
    print(f"Displaying EEF pose data for step {step}")
    
def update_joint_states_only(step):
    clear_output(wait=True)
    visualize_joint_states(step)
    print(f"Displaying joint state data for step {step}")

def update_action_eef_only(step):
    clear_output(wait=True)
    visualize_action_eef(step)
    print(f"Displaying action EEF data for step {step}")
    
def update_action_joints_only(step):
    clear_output(wait=True)
    visualize_action_joints(step)
    print(f"Displaying action joint data for step {step}")

# RGB slider
rgb_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_steps-1,
    step=1,
    description='RGB Timestep:',
    style={'description_width': 'initial'}
)

# Depth slider
depth_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_steps-1,
    step=1,
    description='Depth Timestep:',
    style={'description_width': 'initial'}
)

# Point cloud slider
pc_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_steps-1,
    step=1,
    description='PC Timestep:',
    style={'description_width': 'initial'}
)

# EEF pose slider
eef_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_steps-1,
    step=1,
    description='EEF Pose Timestep:',
    style={'description_width': 'initial'}
)

# Joint states slider
joint_states_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_steps-1,
    step=1,
    description='Joint States Timestep:',
    style={'description_width': 'initial'}
)

# Action EEF slider
action_eef_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_steps-1,
    step=1,
    description='Action EEF Timestep:',
    style={'description_width': 'initial'}
)

# Action joints slider
action_joints_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=num_steps-1,
    step=1,
    description='Action Joints Timestep:',
    style={'description_width': 'initial'}
)

print("Use separate sliders for different data types:")
display(widgets.interactive(update_rgb_only, step=rgb_slider))
display(widgets.interactive(update_depth_only, step=depth_slider))
display(widgets.interactive(update_pointcloud_only, step=pc_slider))
display(widgets.interactive(update_eef_pose_only, step=eef_slider))
display(widgets.interactive(update_joint_states_only, step=joint_states_slider))
display(widgets.interactive(update_action_eef_only, step=action_eef_slider))
display(widgets.interactive(update_action_joints_only, step=action_joints_slider))

Use separate sliders for different data types:


interactive(children=(IntSlider(value=0, description='RGB Timestep:', max=31, style=SliderStyle(description_wi…

interactive(children=(IntSlider(value=0, description='Depth Timestep:', max=31, style=SliderStyle(description_…

interactive(children=(IntSlider(value=0, description='PC Timestep:', max=31, style=SliderStyle(description_wid…

interactive(children=(IntSlider(value=0, description='EEF Pose Timestep:', max=31, style=SliderStyle(descripti…

interactive(children=(IntSlider(value=0, description='Joint States Timestep:', max=31, style=SliderStyle(descr…

interactive(children=(IntSlider(value=0, description='Action EEF Timestep:', max=31, style=SliderStyle(descrip…

interactive(children=(IntSlider(value=0, description='Action Joints Timestep:', max=31, style=SliderStyle(desc…

## 关于 third_view_rgb 数据类型和 RGB 转换说明

### 数据类型解释

third_view_rgb 数据集的 dtype 为 `|S74283`，这是一个固定长度的字节字符串类型：
- `|` 表示使用标准大小和对齐方式
- `S` 表示这是一个字节字符串（String）类型
- `74283` 表示每个元素的最大字节长度为 74283 字节

这种数据类型通常用于存储编码后的图像数据，如 JPEG 或 PNG 格式。

### RGB 转换过程

将字节字符串转换为可显示的 RGB 图像的过程如下：

1. 首先将字节字符串转换为 NumPy 数组
2. 使用 OpenCV 解码图像
3. 将 BGR 格式转换为 RGB 格式

代码示例：

```python
# 从 HDF5 文件中读取字节数据
rgb_bytes = file['third_view_rgb'][step]

# 将字节数据转换为 NumPy 数组
nparr = np.frombuffer(rgb_bytes, np.uint8)

# 使用 OpenCV 解码图像
img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

# 将 BGR 转换为 RGB
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
```

这个过程与现有的 `decode_rgb_image` 函数实现一致。